# SmartStock Intelligence Offline Training Pipeline

This notebook trains the offline intelligence models (KMeans clustering, PCA, and Isolation Forest) for the SmartStock Retail Intelligence Platform. 
It saves the trained artifacts into the `models/` directory so they can be loaded by the Streamlit application for runtime inference.

## 1. Environment & Setup
Install required dependencies and configure the Kaggle API to download the reference dataset.

In [ ]:
!pip install -q kaggle pandas numpy scikit-learn matplotlib seaborn joblib plotly

import os
import json
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.ensemble import IsolationForest
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Dataset Acquisition
Dataset: `mragpavank/predicting-the-sales-of-products-of-a-retail-chain`

In [ ]:
import getpass
import zipfile

print("Enter your Kaggle username:")
kaggle_username = getpass.getpass()
print("Enter your Kaggle key:")
kaggle_key = getpass.getpass()

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

with open(os.path.join(kaggle_dir, "kaggle.json"), "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(os.path.join(kaggle_dir, "kaggle.json"), 0o600)

!kaggle datasets download -d mragpavank/predicting-the-sales-of-products-of-a-retail-chain -p ./data --unzip

# Load actual data (we assume train_data.csv provides the historical demand context)
df = pd.read_csv("./data/train_data.csv")
print("Dataset shape:", df.shape)

## 3. Dataset Validation & Merging

In [ ]:
df['date'] = pd.to_datetime(df['date'])
print("Date range:", df['date'].min(), "to", df['date'].max())
print("Total Products:", df['product_identifier'].nunique())
print("Total Outlets:", df['outlet'].nunique())
print("Missing Values:\n", df.isnull().sum())

# Clean negative sales if any
df['sales'] = df['sales'].clip(lower=0)

## 4. Behavioral Feature Engineering for Clustering
We aggregate at the product level to capture overall product behavioral segments.

In [ ]:
def build_product_features(data):
    behavior = data.groupby('product_identifier').agg(
        mean_sales=('sales', 'mean'),
        median_sales=('sales', 'median'),
        sales_std=('sales', 'std'),
        zero_sales_ratio=('sales', lambda x: (x == 0).mean())
    )
    behavior['sales_std'] = behavior['sales_std'].fillna(0)
    behavior['coefficient_of_variation'] = np.where(
        behavior['mean_sales'] == 0, 0, behavior['sales_std'] / behavior['mean_sales']
    )
    return behavior

product_features = build_product_features(df)
CLUSTER_FEATURES = ['mean_sales', 'sales_std', 'coefficient_of_variation', 'zero_sales_ratio']
X = product_features[CLUSTER_FEATURES].copy()
print("Feature Matrix Shape:", X.shape)

## 5. Scaling & Preprocessing

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ensure models dir exists
os.makedirs("../models/clustering", exist_ok=True)
joblib.dump(scaler, "../models/clustering/product_scaler.joblib")
print("Scaler saved.")

## 6. K Selection
We evaluate multiple K values using Inertia and Silhouette Score.

In [ ]:
k_values = range(2, min(8, len(X)))
inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, ax1 = plt.subplots()
ax1.plot(k_values, inertias, 'b-o', label='Inertia')
ax1.set_xlabel('Number of clusters (K)')
ax1.set_ylabel('Inertia', color='b')

ax2 = ax1.twinx()
ax2.plot(k_values, sil_scores, 'r-s', label='Silhouette')
ax2.set_ylabel('Silhouette Score', color='r')

plt.title('K Selection: Inertia vs Silhouette')
plt.show()

selected_k = k_values[np.argmax(sil_scores)]
print(f"Selected optimal K: {selected_k} based on highest Silhouette Score ({max(sil_scores):.4f})")

## 7. Final KMeans Training & PCA

In [ ]:
final_kmeans = KMeans(n_clusters=selected_k, random_state=42, n_init="auto")
product_features['cluster'] = final_kmeans.fit_predict(X_scaled)
joblib.dump(final_kmeans, "../models/clustering/product_kmeans.joblib")
print("KMeans model saved.")

pca = PCA(n_components=2)
pca_coords = pca.fit_transform(X_scaled)
product_features['pca_1'] = pca_coords[:, 0]
product_features['pca_2'] = pca_coords[:, 1]

joblib.dump(pca, "../models/clustering/product_pca.joblib")
print("PCA model saved.")

## 8. Cluster Profiling
We generate deterministic profiles for each cluster based on summary stats.

In [ ]:
profiles = {}
for c in range(selected_k):
    subset = product_features[product_features['cluster'] == c]
    mean_vol = subset['mean_sales'].mean()
    mean_cv = subset['coefficient_of_variation'].mean()
    
    # Simple deterministic naming logic based on global medians
    vol_label = "High Volume" if mean_vol > product_features['mean_sales'].median() else "Low Volume"
    var_label = "High Volatility" if mean_cv > product_features['coefficient_of_variation'].median() else "Low Volatility"
    
    profiles[str(c)] = {
        "name": f"{vol_label} / {var_label}",
        "product_count": int(len(subset)),
        "mean_sales": float(mean_vol),
        "sales_std": float(subset['sales_std'].mean()),
        "coefficient_of_variation": float(mean_cv)
    }

with open("../models/clustering/cluster_profiles.json", "w") as f:
    json.dump(profiles, f, indent=4)

with open("../models/clustering/clustering_features.json", "w") as f:
    json.dump({"features": CLUSTER_FEATURES}, f, indent=4)

print("Profiles saved.")

## 9. Product Similarity Representation
We save the scaled behavior vectors to rapidly compute cosine similarities during inference.

In [ ]:
joblib.dump(X_scaled, "../models/clustering/product_behavior_matrix.joblib")
with open("../models/clustering/product_index.json", "w") as f:
    json.dump({"products": product_features.index.tolist()}, f, indent=4)
print("Similarity artifacts saved.")

## 10. Anomaly Detection Training

In [ ]:
os.makedirs("../models/anomaly", exist_ok=True)

anomaly_df = df.sort_values(by=['outlet', 'product_identifier', 'date']).copy()
anomaly_df['rolling_mean_14'] = anomaly_df.groupby(['outlet', 'product_identifier'])['sales'].transform(lambda x: x.rolling(14, min_periods=1).mean())
anomaly_df['rolling_std_14'] = anomaly_df.groupby(['outlet', 'product_identifier'])['sales'].transform(lambda x: x.rolling(14, min_periods=1).std().fillna(0))

ANOMALY_FEATURES = ['sales', 'rolling_mean_14', 'rolling_std_14']
valid_idx = anomaly_df[ANOMALY_FEATURES].dropna().index
X_anomaly = anomaly_df.loc[valid_idx, ANOMALY_FEATURES]

iso = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_anomaly)

joblib.dump(iso, "../models/anomaly/isolation_forest.joblib")
with open("../models/anomaly/anomaly_features.json", "w") as f:
    json.dump({"features": ANOMALY_FEATURES}, f, indent=4)

print("Isolation Forest anomaly model saved.")

## 11. Artifact Reload & Inference Verification
Test that we can load the models and predict correctly.

In [ ]:
del scaler, final_kmeans, pca, iso

test_scaler = joblib.load("../models/clustering/product_scaler.joblib")
test_kmeans = joblib.load("../models/clustering/product_kmeans.joblib")
test_pca = joblib.load("../models/clustering/product_pca.joblib")
test_iso = joblib.load("../models/anomaly/isolation_forest.joblib")

# Validate
test_scaled = test_scaler.transform(X)
test_labels = test_kmeans.predict(test_scaled)
test_coords = test_pca.transform(test_scaled)

assert np.allclose(test_scaled, X_scaled), "Scaler reload mismatch"
assert np.array_equal(test_labels, product_features['cluster']), "KMeans reload mismatch"
assert np.allclose(test_coords[:, 0], product_features['pca_1']), "PCA reload mismatch"

print("SUCCESS! All offline intelligence artifacts have been verified and saved to `models/`.")